# Train CNN
*This notebook aims to train a CNN model to classify images (pokemon 2D pictures)*

## Summary

- [Imports & Configuration](#imports--configuration)
- [Data Preparation](#data-preparation)
- [Model](#model)
- [Optimizer and Criterion](#optimizer-and-criterion)
- [Training with logging MLFlow](#training-with-loggin-mlflow)

## Imports & Configuration

In [1]:
import os
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from dotenv import load_dotenv, find_dotenv

# Recherche automatique du .env dans ton arborescence
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError(".env introuvable – place-le à la racine du projet")

print("→ Chargement de", dotenv_path)
load_dotenv(dotenv_path)

# Debug : affiche ce qui a été chargé
print("MLFLOW_TRACKING_URI   =", os.getenv("MLFLOW_TRACKING_URI"))
print("MLFLOW_TRACKING_USER  =", os.getenv("MLFLOW_TRACKING_USERNAME"))
# on ne print pas la password par sécurité, mais vérifie qu'elle existe
assert os.getenv("MLFLOW_TRACKING_PASSWORD"), "MLFLOW_TRACKING_PASSWORD n'est pas défini"

from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=os.getenv("MLFLOW_TRACKING_URI"))
exp = client.get_experiment_by_name("final-project")
if exp is None:
    exp_id = client.create_experiment("final-project")
    print(f"✅ Créé l’expériment ‘final-project’ avec ID {exp_id}")
else:
    print(f"ℹ️ Expérience existante (ID {exp.experiment_id})")

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("final-project")

# Adaptation pour AMD via DirectML
try:
    import torch_directml
    device = torch_directml.device()
    print("Detected DirectML device")
except ImportError:
    # fallback habituel
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

print(f"Using device: {device}")


→ Chargement de c:\Users\marco\Desktop\devops-pokefind\.env
MLFLOW_TRACKING_URI   = https://dagshub.com/Marco2a94/final-project.mlflow
MLFLOW_TRACKING_USER  = Marco2a94
ℹ️ Expérience existante (ID 0)
Detected DirectML device
Using device: privateuseone:0


## Data Preparation

In [2]:
from pathlib import Path
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# Recherche du dossier data/raw dans les parents
cwd = Path.cwd()
raw_dir = None
for ancestor in [cwd] + list(cwd.parents):
    candidate = ancestor / "data" / "raw"
    if candidate.is_dir():
        raw_dir = candidate
        break

if raw_dir is None:
    raise FileNotFoundError(f"Impossible de trouver le dossier 'data/raw' depuis {cwd}")

print(f"→ RAW_DIR détecté : {raw_dir}")

# Hyper-paramètres
BATCH_SIZE  = 32
IMG_SIZE    = 224
TRAIN_RATIO = 0.8
SEED        = 42

# Transforms
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Chargement complet
full_dataset = ImageFolder(str(raw_dir))
class_names  = full_dataset.classes
n_total      = len(full_dataset)
n_train      = int(n_total * TRAIN_RATIO)
n_val        = n_total - n_train

# Split aléatoire
torch.manual_seed(SEED)
train_subset, val_subset = random_split(full_dataset, [n_train, n_val])

# Assigner les transforms respectifs
train_subset.dataset.transform = train_transforms
val_subset.dataset.transform   = val_transforms

# Création des DataLoaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Total images: {n_total} → Train: {n_train}, Val: {n_val}")
print(f"Classes détectées: {class_names}")


→ RAW_DIR détecté : c:\Users\marco\Desktop\devops-pokefind\data\raw
Total images: 10657 → Train: 8525, Val: 2132
Classes détectées: ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine', 'Articuno', 'Beedrill', 'Bellsprout', 'Blastoise', 'Bulbasaur', 'Butterfree', 'Caterpie', 'Chansey', 'Charizard', 'Charmander', 'Charmeleon', 'Clefable', 'Clefairy', 'Cloyster', 'Cubone', 'Dewgong', 'Diglett', 'Ditto', 'Dodrio', 'Doduo', 'Dragonair', 'Dragonite', 'Dratini', 'Drowzee', 'Dugtrio', 'Eevee', 'Ekans', 'Electabuzz', 'Electrode', 'Exeggcute', 'Exeggutor', 'Farfetchd', 'Fearow', 'Flareon', 'Gastly', 'Gengar', 'Geodude', 'Gloom', 'Golbat', 'Goldeen', 'Golduck', 'Golem', 'Graveler', 'Grimer', 'Growlithe', 'Gyarados', 'Haunter', 'Hitmonchan', 'Hitmonlee', 'Horsea', 'Hypno', 'Ivysaur', 'Jigglypuff', 'Jolteon', 'Jynx', 'Kabuto', 'Kabutops', 'Kadabra', 'Kakuna', 'Kangaskhan', 'Kingler', 'Koffing', 'Krabby', 'Lapras', 'Lickitung', 'Machamp', 'Machoke', 'Machop', 'Magikarp', 'Magmar', 'Magnemite', 'M

## Model

In [3]:
from torchvision import models

# Charger un ResNet18 pré-entraîné
model = models.resnet18(pretrained=True)

# Récupérer le nombre de features de la dernière couche
num_ftrs = model.fc.in_features

# Adapter la dernière couche au nombre de classes détectées
# class_names vient de la cellule 2 : full_dataset.classes
model.fc = nn.Linear(num_ftrs, len(class_names))

# Envoyer sur GPU/CPU
model = model.to(device)

model = torch.compile(model)

print(model)


c:\Users\marco\Desktop\devops-pokefind\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\marco\Desktop\devops-pokefind\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


OptimizedModule(
  (_orig_mod): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True,

## Optimizer and Criterion

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


## Training with loggin MLFlow

In [ ]:
import torch._dynamo as dynamo
import torch
from torch.cuda.amp import autocast, GradScaler

# ─────────────────────────────────────────────────
# 0) Désactivation de TorchDynamo pour forcer l'Eager
dynamo.config.suppress_errors = True
dynamo.disable()
print("🔒 TorchDynamo désactivé, mode Eager activé")

# 1) Préparation AMP
scaler = GradScaler()

EPOCHS = 5
MODEL_PATH = "model.pth"

with mlflow.start_run():
    # Logging des hyper-params
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("epochs",     EPOCHS)
    mlflow.log_param("lr",         1e-3)
    mlflow.log_param("model_arch", "resnet18")

    # Boucle d'entraînement + validation
    for epoch in range(1, EPOCHS + 1):
        # — Training —
        model.train()
        running_loss = correct = total = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # — Validation —
        model.eval()
        val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                with autocast():
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc   = val_correct / val_total

        # Logging des métriques
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc",  train_acc,  step=epoch)
        mlflow.log_metric("val_loss",   val_loss,   step=epoch)
        mlflow.log_metric("val_acc",    val_acc,    step=epoch)

        print(f"Epoch {epoch}/{EPOCHS} • "
              f"Train: loss={train_loss:.4f}, acc={train_acc:.4f} • "
              f"Val:   loss={val_loss:.4f}, acc={val_acc:.4f}")

    # ─────────────────────────────────────────────
    # 2) Sauvegarde manuelle et log artefact
    torch.save(model.state_dict(), MODEL_PATH)
    mlflow.log_artifact(MODEL_PATH, artifact_path="model")
    print(f"✅ Modèle sauvegardé dans '{MODEL_PATH}' et loggé en tant qu'artefact")


🔒 TorchDynamo désactivé, mode Eager activé


C:\Users\marco\AppData\Local\Temp\ipykernel_24304\1266762133.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
c:\Users\marco\Desktop\devops-pokefind\.venv\lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
C:\Users\marco\AppData\Local\Temp\ipykernel_24304\1266762133.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\marco\Desktop\devops-pokefind\.venv\lib\site-packages\torch\amp\autocast_mode.py:265: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
W0710 20:23:17.544000 27228 torch\_dynamo\convert_frame.py:1009] WON'T CONVERT forward c:\Users\marco\Desktop\devops-pokefind\.venv\lib\site-packages\torchvision\models\res

Epoch 1/5 • Train: loss=2.8535, acc=0.3519 • Val:   loss=2.0118, acc=0.4991
Epoch 2/5 • Train: loss=1.2640, acc=0.6666 • Val:   loss=1.7132, acc=0.5797
Epoch 3/5 • Train: loss=0.7419, acc=0.7970 • Val:   loss=1.4717, acc=0.6417
Epoch 4/5 • Train: loss=0.4689, acc=0.8709 • Val:   loss=1.3984, acc=0.6717


2025/07/10 20:29:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 5/5 • Train: loss=0.2998, acc=0.9186 • Val:   loss=1.2464, acc=0.7120
🏃 View run tasteful-grub-480 at: https://dagshub.com/Marco2a94/final-project.mlflow/#/experiments/0/runs/f9fb3560aca34ffaa05291449a15fd23
🧪 View experiment at: https://dagshub.com/Marco2a94/final-project.mlflow/#/experiments/0


RestException: INTERNAL_ERROR: Response: {'error': 'unsupported endpoint, please contact support@dagshub.com'}

## Test and Visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Charger le modèle loggé dans MLflow (téléchargé localement via mlruns, ou directement depuis DagsHub)
run_id = mlflow.active_run().info.run_id
model_uri = f"runs:/{run_id}/model"
loaded_model = mlflow.pytorch.load_model(model_uri).to(device)
loaded_model.eval()

# Prendre un batch de validation
imgs, labels = next(iter(val_loader))
imgs, labels = imgs.to(device), labels.to(device)

# Faire la prédiction
with torch.no_grad():
    outputs = loaded_model(imgs)
    _, preds = torch.max(outputs, 1)

# Préparer la grille d’images (4 premières)
n_display = min(4, imgs.size(0))
fig, axes = plt.subplots(1, n_display, figsize=(12, 3))
for i in range(n_display):
    ax = axes[i]
    img = imgs[i].cpu().permute(1, 2, 0).numpy()
    # dé-normalisation ImageNet
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f"P: {class_names[preds[i]]}\nT: {class_names[labels[i]]}")
    ax.axis("off")
plt.show()
